
# 02. Queues and Latency Under Load
**Goal:** Visualize how queue depth and latency evolve under varying load conditions.


In [5]:

import sys
import os
import time
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Basic configuration
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Add project root to path
# Assuming we are in artemis_final/notebooks/load_balancer/
current_dir = Path(os.getcwd())
project_root = current_dir.parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from artemis_final.load_balancer.public_api import (
    ArtemisLoadBalancer,
    ModelCapacityConfig,
    StatsRegistry,
    RouterOutput,
    SchedulingContext,
)



## 1. Setup Simulation Environment
We define 3 models with different service times:
- **Fast**: 100ms
- **Medium**: 300ms
- **Slow**: 600ms


In [6]:
def setup_environment(replicas=1):
    stats = StatsRegistry()
    task = "vqa"
    
    model_stats = {
        "fast_model": (100.0, 0.9, 0.002),
        "medium_model": (300.0, 0.92, 0.005),
        "slow_model": (600.0, 0.95, 0.01),
    }

    for m, (lat, acc, cost) in model_stats.items():
        stats.update_latency(task, m, lat)
        stats.update_accuracy(task, m, acc)
        stats.update_cost(task, m, cost)

    configs = {}
    for m, (lat, acc, cost) in model_stats.items():
        configs[m] = ModelCapacityConfig(
            model_name=m,
            base_latency_ms=lat,
            min_replicas=replicas,
            max_replicas=replicas,
            max_qps_per_replica=20.0,
            cost_per_request_usd=cost
        )
    
    # Loose SLA so we can observe queues building up without just rejecting
    sla = {"default": 5000.0} 
    
    lb = ArtemisLoadBalancer(
        model_configs=configs,
        stats_registry=stats,
        latency_sla_ms=sla,
        scheduling_mode="capacity_aware",
        simulation_only=True
    )
    
    return lb

print("Environment setup function defined.")

Environment setup function defined.



## 2. Simulation Loop
A function to simulate N requests arriving at a fixed interval.


In [7]:

def run_simulation(lb, n_requests=200, inter_arrival_ms=100.0):
    decisions = []
    current_time = time.time()
    
    # Constant router preference for 'medium_model'
    # but we give some probability to others too
    router_out = RouterOutput(
        router_probs={"fast_model": 0.2, "medium_model": 0.6, "slow_model": 0.2},
        preferred_model="medium_model",
        max_prob=0.6,
        sample_id='sim_req', task_type='vqa',
    )
    
    for i in range(n_requests):
        # Advance time
        current_time += (inter_arrival_ms / 1000.0)
        
        ctx = SchedulingContext(
            arrival_ts_ms=current_time * 1000,
            sample_id=f"req_{i}",
            task_type="vqa"
        )
        
        decision = lb.schedule(router_out, ctx)
        
        decisions.append({
            "req_id": i,
            "arrival_time": current_time,
            "chosen_model": decision.chosen_model,
            "queue_delay": decision.queue_delay_ms,
            "service_time": decision.service_time_ms,
            "total_latency": decision.total_latency_ms
        })
        
    return pd.DataFrame(decisions)



## 3. Scenario A: Low Load
Inter-arrival time (200ms) > Fast Model Latency (100ms), but < Slow Model (600ms).
Ideally, the LB should handle this well by distributing load.


In [8]:

lb_low = setup_environment(replicas=1)
df_low = run_simulation(lb_low, n_requests=100, inter_arrival_ms=300.0)

plt.figure(figsize=(10, 4))
plt.plot(df_low['req_id'], df_low['total_latency'], label='Total Latency', marker='.')
plt.plot(df_low['req_id'], df_low['queue_delay'], label='Queue Delay', alpha=0.5)
plt.title("Latency Context: Low Load (300ms interval)")
plt.ylabel("Time (ms)")
plt.xlabel("Request ID")
plt.legend()
plt.show()

print("Average Queue Delay:", df_low['queue_delay'].mean())


TypeError: RouterOutput.__init__() missing 2 required positional arguments: 'sample_id' and 'task_type'


## 4. Scenario B: High Load (Overload)
Inter-arrival time (50ms) is much faster than even the fastest model (100ms).
Queues should explode linearly.


In [ ]:

lb_high = setup_environment(replicas=1)
df_high = run_simulation(lb_high, n_requests=100, inter_arrival_ms=50.0)

plt.figure(figsize=(10, 4))
plt.plot(df_high['req_id'], df_high['total_latency'], label='Total Latency', color='red')
plt.plot(df_high['req_id'], df_high['queue_delay'], label='Queue Delay', ls='--', color='orange')
plt.title("Latency Context: High Load (50ms interval - Overload)")
plt.ylabel("Time (ms)")
plt.xlabel("Request ID")
plt.legend()
plt.show()



## 5. Model Usage Distribution
Let's see which models handled the traffic in the high load scenario.


In [ ]:

usage = df_high['chosen_model'].value_counts()
usage.plot(kind='bar', color=['teal', 'coral', 'gold'])
plt.title("Model Assignment Count (High Load)")
plt.ylabel("Requests")
plt.show()
